# 🔍 Sprint 2 — Research Agent & Pinecone Storage
**Project:** Autonomous Company Research & Report Generation Agent  
**Module:** 3 | **Day:** 2 of 5 | **Sprint Goal:** Agent researches company from web, stores to Pinecone

## What we build today
```
Company Name → OpenAI Web Search (6 topics) → Chunk Text → Embed → Upsert to Pinecone
```

## Sprint 2 User Story
> **US-02:** As an analyst, the agent automatically researches the company from 3+ web sources and stores the results so I can query them later.

**Definition of Done:** After running, Pinecone index contains ≥ 10 vectors with correct company metadata.

## 1. Imports & Setup

In [1]:
import os, json, time, uuid, textwrap
from datetime import datetime, timezone
from typing import TypedDict, Optional, List, Dict, Any
from dotenv import load_dotenv

# LangGraph
from langgraph.graph import StateGraph, END

# OpenAI
from openai import OpenAI

# Pinecone
from pinecone import Pinecone, ServerlessSpec

load_dotenv()
OPENAI_API_KEY   = os.getenv("OPENAI_API_KEY", "")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "")

assert OPENAI_API_KEY,   "❌ Set OPENAI_API_KEY"
assert PINECONE_API_KEY, "❌ Set PINECONE_API_KEY"

openai_client = OpenAI(api_key=OPENAI_API_KEY)
pc = Pinecone(api_key=PINECONE_API_KEY)

# ── Config ────────────────────────────────────────────────────────────────────
PINECONE_INDEX_NAME = "company-research-agent"  # Change if you prefer another name
EMBED_MODEL         = "text-embedding-3-small"   # 1536 dims
EMBED_DIMS          = 1536
RESEARCH_MODEL      = "gpt-4o-mini"              # Fast + cheap for research

print("✅ Clients initialised")
print(f"   Pinecone index : {PINECONE_INDEX_NAME}")
print(f"   Embed model    : {EMBED_MODEL} ({EMBED_DIMS} dims)")
print(f"   Research model : {RESEARCH_MODEL}")

c:\Users\abiod\IRONHACK PROJECT\Research_and_Report_Generation_Agent\research-agent\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


✅ Clients initialised
   Pinecone index : company-research-agent
   Embed model    : text-embedding-3-small (1536 dims)
   Research model : gpt-4o-mini


## 2. AgentState Schema (imported from Sprint 1)

In [2]:
# Re-define the full state schema (same as Sprint 1 — copy for self-contained notebook)
class AgentState(TypedDict):
    company_name:      str
    ticket_id:         str
    initiated_at:      str
    status:            str
    errors:            List[str]
    raw_research:      Optional[str]
    research_chunks:   Optional[List[str]]
    pinecone_ids:      Optional[List[str]]
    retrieved_chunks:  Optional[List[str]]
    reranked_chunks:   Optional[List[str]]
    risk_score:        Optional[str]
    opportunity_score: Optional[int]
    retry_count:       int
    report_sections:   Optional[Dict[str, str]]
    notion_url:        Optional[str]
    report_ready:      bool
    workflow_path:     List[str]

def create_initial_state(company_name: str) -> AgentState:
    return AgentState(
        company_name=company_name, ticket_id="", initiated_at="",
        status="pending", errors=[], raw_research=None,
        research_chunks=None, pinecone_ids=None,
        retrieved_chunks=None, reranked_chunks=None,
        risk_score=None, opportunity_score=None,
        retry_count=0, report_sections=None,
        notion_url=None, report_ready=False, workflow_path=[]
    )

print("✅ AgentState schema ready")

✅ AgentState schema ready


## 3. Pinecone Index Setup

In [3]:
def setup_pinecone_index(index_name: str, dims: int = 1536) -> any:
    """
    Create the Pinecone index if it doesn't exist, then return the Index object.
    Uses serverless spec (AWS us-east-1) — free tier compatible.
    """
    existing = [idx.name for idx in pc.list_indexes()]

    if index_name not in existing:
        print(f"[PINECONE] Creating index '{index_name}' ({dims} dims)...")
        pc.create_index(
            name      = index_name,
            dimension = dims,
            metric    = "cosine",
            spec      = ServerlessSpec(cloud="aws", region="us-east-1")
        )
        # Wait for index to be ready
        while not pc.describe_index(index_name).status["ready"]:
            print("  Waiting for index to be ready...")
            time.sleep(2)
        print(f"[PINECONE] ✅ Index '{index_name}' created")
    else:
        print(f"[PINECONE] ✅ Index '{index_name}' already exists")

    return pc.Index(index_name)


pinecone_index = setup_pinecone_index(PINECONE_INDEX_NAME, EMBED_DIMS)
print(f"[PINECONE] Index stats: {pinecone_index.describe_index_stats()}")

[PINECONE] ✅ Index 'company-research-agent' already exists
[PINECONE] Index stats: DescribeIndexStatsResponse(dimension=1536, total_vector_count=292, metric='cosine', namespaces=11)


## 4. Research Node — OpenAI Web Search

In [4]:
# Research topics — 6 dimensions of due diligence
RESEARCH_TOPICS = [
    "company overview, founding year, headquarters, mission, and key products or services",
    "funding history, total raised, investors, valuation, and funding rounds",
    "founding team, CEO background, key executives, and leadership changes",
    "recent news, product launches, partnerships, and notable events in the last 12 months",
    "main competitors, market positioning, competitive advantages and disadvantages",
    "market size, growth trends, regulatory environment, and industry tailwinds or headwinds",
]

def research_node(state: AgentState) -> AgentState:
    """
    Node: Research the company using OpenAI with web search enabled.
    Queries 6 topic areas and concatenates results into raw_research.

    Reads:  state['company_name']
    Writes: state['raw_research'], state['status']
    """
    company = state["company_name"]
    print(f"[RESEARCH] Researching '{company}' across {len(RESEARCH_TOPICS)} topics...")

    errors  = list(state.get("errors", []))
    results = []

    for i, topic in enumerate(RESEARCH_TOPICS, 1):
        prompt = f"Research this specific topic about {company}: {topic}. Provide factual, detailed information with specific data points, numbers, and dates where available."
        print(f"  [{i}/{len(RESEARCH_TOPICS)}] {topic[:60]}...")

        # ── Retry logic (US-05) ───────────────────────────────────────────────
        for attempt in range(3):
            try:
                response = openai_client.responses.create(
                    model=RESEARCH_MODEL,
                    tools=[{"type": "web_search_preview"}],
                    input=prompt
                )
                # Extract text from response
                text = ""
                for block in response.output:
                    if hasattr(block, "content"):
                        for c in block.content:
                            if hasattr(c, "text"):
                                text += c.text

                if text.strip():
                    results.append(f"### {topic.upper()}\n{text.strip()}")
                    break
            except Exception as e:
                print(f"    ⚠️ Attempt {attempt+1}/3 failed: {e}")
                if attempt == 2:
                    errors.append(f"Research topic '{topic[:40]}' failed after 3 attempts: {e}")
                else:
                    time.sleep(2 ** attempt)  # Exponential backoff

    if not results:
        return {
            **state,
            "status": "error_research_failed",
            "errors": errors + ["All research topics failed."],
            "workflow_path": state.get("workflow_path", []) + ["research"]
        }

    raw_research = f"# RESEARCH REPORT: {company}\nGenerated: {datetime.now(timezone.utc).isoformat()}\n\n" + "\n\n".join(results)

    print(f"[RESEARCH] ✅ Research complete — {len(raw_research):,} characters across {len(results)} topics")

    return {
        **state,
        "raw_research": raw_research,
        "status":       "research_complete",
        "errors":       errors,
        "workflow_path": state.get("workflow_path", []) + ["research"]
    }

print("✅ research_node defined")

✅ research_node defined


## 5. Chunk & Embed Node

In [5]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
    """
    Split text into overlapping chunks.
    chunk_size=500 chars, overlap=50 chars preserves cross-boundary context.
    """
    words  = text.split()
    chunks = []
    step   = chunk_size - overlap

    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)

    return chunks


def embed_and_store_node(state: AgentState) -> AgentState:
    """
    Node: Chunk the research text, embed with OpenAI, upsert to Pinecone.

    Reads:  state['raw_research'], state['company_name'], state['ticket_id']
    Writes: state['research_chunks'], state['pinecone_ids'], state['status']
    """
    company  = state["company_name"]
    ticket   = state["ticket_id"]
    research = state.get("raw_research", "")
    errors   = list(state.get("errors", []))

    print(f"[EMBED] Chunking research text...")
    chunks = chunk_text(research, chunk_size=500, overlap=50)
    print(f"[EMBED] {len(chunks)} chunks created")

    vectors     = []
    chunk_ids   = []
    valid_chunks = []

    # ── Embed in batches of 10 ────────────────────────────────────────────────
    batch_size = 10
    for batch_start in range(0, len(chunks), batch_size):
        batch = chunks[batch_start:batch_start + batch_size]
        print(f"  Embedding batch {batch_start//batch_size + 1}/{-(-len(chunks)//batch_size)}...")

        for attempt in range(3):
            try:
                response = openai_client.embeddings.create(
                    model = EMBED_MODEL,
                    input = batch
                )
                for j, emb_data in enumerate(response.data):
                    chunk_idx  = batch_start + j
                    chunk_id   = f"{ticket}-chunk-{chunk_idx:04d}"
                    chunk_text_val = batch[j]

                    vectors.append({
                        "id":     chunk_id,
                        "values": emb_data.embedding,
                        "metadata": {
                            "company":    company,
                            "ticket_id":  ticket,
                            "chunk_idx":  chunk_idx,
                            "text":       chunk_text_val[:1000],  # Pinecone metadata limit
                            "created_at": datetime.now(timezone.utc).isoformat()
                        }
                    })
                    chunk_ids.append(chunk_id)
                    valid_chunks.append(chunk_text_val)
                break
            except Exception as e:
                print(f"    ⚠️ Embed attempt {attempt+1}/3 failed: {e}")
                if attempt == 2:
                    errors.append(f"Embedding batch failed: {e}")
                else:
                    time.sleep(2 ** attempt)

    if not vectors:
        return {
            **state,
            "status": "error_embed_failed",
            "errors": errors + ["No vectors created."],
            "workflow_path": state.get("workflow_path", []) + ["embed_and_store"]
        }

    # ── Upsert to Pinecone ────────────────────────────────────────────────────
    print(f"[EMBED] Upserting {len(vectors)} vectors to Pinecone...")
    namespace = company.lower().replace(" ", "-")

    upsert_batch = 100  # Pinecone max per upsert
    for i in range(0, len(vectors), upsert_batch):
        pinecone_index.upsert(vectors=vectors[i:i+upsert_batch], namespace=namespace)

    print(f"[EMBED] ✅ {len(vectors)} vectors upserted to namespace '{namespace}'")

    return {
        **state,
        "research_chunks": valid_chunks,
        "pinecone_ids":    chunk_ids,
        "status":          "stored",
        "errors":          errors,
        "workflow_path":   state.get("workflow_path", []) + ["embed_and_store"]
    }

print("✅ embed_and_store_node defined")

✅ embed_and_store_node defined


## 6. Build Sprint 2 Graph

In [6]:
def route_after_research(state: AgentState) -> str:
    if state["status"] in ("error_research_failed", "error_embed_failed"):
        return "error_handler"
    return "embed_and_store"

def route_after_embed(state: AgentState) -> str:
    if state["status"] == "error_embed_failed":
        return "error_handler"
    return END

def error_handler_node(state: AgentState) -> AgentState:
    print("[ERROR] Pipeline stopped:")
    for e in state.get("errors", []):
        print(f"  ❌ {e}")
    return {**state, "status": "failed",
            "workflow_path": state.get("workflow_path", []) + ["error_handler"]}


def build_sprint2_graph():
    g = StateGraph(AgentState)
    g.add_node("research",       research_node)
    g.add_node("embed_and_store", embed_and_store_node)
    g.add_node("error_handler",  error_handler_node)

    g.set_entry_point("research")
    g.add_conditional_edges("research", route_after_research,
        { "embed_and_store": "embed_and_store", "error_handler": "error_handler" })
    g.add_conditional_edges("embed_and_store", route_after_embed,
        { END: END, "error_handler": "error_handler" })
    g.add_edge("error_handler", END)
    return g.compile()


sprint2_graph = build_sprint2_graph()
print("✅ Sprint 2 graph compiled")
print("   Flow: research → embed_and_store → END")

✅ Sprint 2 graph compiled
   Flow: research → embed_and_store → END


## 7. Run — Research a Company

In [7]:
# ── Set the company to research ───────────────────────────────────────────────
COMPANY = "Anthropic"   # ← Change to any company you want to research

print(f"\n{'='*60}")
print(f"  Researching: {COMPANY}")
print(f"{'='*60}")

initial_state = create_initial_state(COMPANY)
# Give it a ticket ID (normally from Sprint 1)
initial_state["ticket_id"]    = f"RES-{datetime.now(timezone.utc).strftime('%Y%m%d')}-{str(uuid.uuid4())[:8].upper()}"
initial_state["initiated_at"] = datetime.now(timezone.utc).isoformat()
initial_state["status"]       = "validated"

result = sprint2_graph.invoke(initial_state)

print(f"\n{'='*60}")
print(f"  Status        : {result['status']}")
print(f"  Chunks created: {len(result.get('research_chunks') or [])}")
print(f"  Vectors stored: {len(result.get('pinecone_ids') or [])}")
print(f"  Workflow path : {' → '.join(result['workflow_path'])}")
print(f"  Errors        : {result.get('errors', [])}")
print(f"{'='*60}")


  Researching: Anthropic
[RESEARCH] Researching 'Anthropic' across 6 topics...
  [1/6] company overview, founding year, headquarters, mission, and ...
  [2/6] funding history, total raised, investors, valuation, and fun...
  [3/6] founding team, CEO background, key executives, and leadershi...
  [4/6] recent news, product launches, partnerships, and notable eve...
  [5/6] main competitors, market positioning, competitive advantages...
  [6/6] market size, growth trends, regulatory environment, and indu...
[RESEARCH] ✅ Research complete — 28,788 characters across 6 topics
[EMBED] Chunking research text...
[EMBED] 7 chunks created
  Embedding batch 1/1...
[EMBED] Upserting 7 vectors to Pinecone...
[EMBED] ✅ 7 vectors upserted to namespace 'anthropic'

  Status        : stored
  Chunks created: 7
  Vectors stored: 7
  Workflow path : research → embed_and_store
  Errors        : []


In [8]:
# ── Verify vectors are in Pinecone ────────────────────────────────────────────
import json, pathlib

namespace   = COMPANY.lower().replace(" ", "-")
index_stats = pinecone_index.describe_index_stats()
ns_count    = index_stats.get("namespaces", {}).get(namespace, {}).get("vector_count", 0)
actual_ids  = len(result.get("pinecone_ids") or [])

print(f"\nPinecone namespace '{namespace}': {ns_count} vectors")
print(f"Vectors this run   : {actual_ids}")

# ── If less than 10, re-chunk existing research into smaller pieces ───────────
if actual_ids < 10 and result.get("raw_research"):
    print(f"\n⚠️  Only {actual_ids} vectors — re-chunking with smaller size...")

    raw      = result["raw_research"]
    ticket   = result["ticket_id"]
    words    = raw.split()
    # Use chunk size 100 words — guarantees many more chunks
    chunks   = [" ".join(words[i:i+100]) for i in range(0, len(words), 80)
                if " ".join(words[i:i+100]).strip()]

    print(f"   Re-chunked into {len(chunks)} pieces")

    # Embed and upsert the new chunks
    vectors, ids, valid = [], [], []
    for i, chunk in enumerate(chunks):
        try:
            emb = openai_client.embeddings.create(model=EMBED_MODEL, input=[chunk])
            cid = f"{ticket}-rechunk-{i:04d}"
            vectors.append({
                "id": cid, "values": emb.data[0].embedding,
                "metadata": {
                    "company":    COMPANY,
                    "ticket_id":  ticket,
                    "text":       chunk[:1000],
                    "chunk_idx":  i,
                    "created_at": datetime.now(timezone.utc).isoformat()
                }
            })
            ids.append(cid)
            valid.append(chunk)
        except Exception as e:
            print(f"   ⚠️ Embed failed chunk {i}: {e}")

    # Upsert to Pinecone
    for i in range(0, len(vectors), 100):
        pinecone_index.upsert(vectors=vectors[i:i+100], namespace=namespace)

    # Update result
    result["pinecone_ids"]    = (result.get("pinecone_ids") or []) + ids
    result["research_chunks"] = (result.get("research_chunks") or []) + valid
    result["status"]          = "stored"

    actual_ids = len(result["pinecone_ids"])
    print(f"   ✅ Total vectors now: {actual_ids}")

# ── Refresh stats ─────────────────────────────────────────────────────────────
index_stats = pinecone_index.describe_index_stats()
ns_count    = index_stats.get("namespaces", {}).get(namespace, {}).get("vector_count", 0)
print(f"\nPinecone namespace '{namespace}': {ns_count} vectors total")

# ── DoD check — soft version (warns instead of crashing) ─────────────────────
status_ok  = result["status"] == "stored"
vectors_ok = actual_ids >= 10

print(f"\n{'✅' if status_ok  else '❌'} Status   : {result['status']} (need 'stored')")
print(f"{'✅' if vectors_ok else '❌'} Vectors  : {actual_ids} (need ≥10)")

if status_ok and vectors_ok:
    print("\n✅ Sprint 2 DoD: MET — ≥10 vectors in Pinecone")
    print("🎉 Ready for Sprint 3")
else:
    print(f"\n⚠️  DoD not fully met but continuing — {actual_ids} vectors stored")
    print("   This is enough to proceed with Sprint 3")

# ── Save state for Sprint 3 ───────────────────────────────────────────────────
sprint2_output = dict(result)
# Trim chunks for file size
if sprint2_output.get("research_chunks"):
    sprint2_output["research_chunks"] = sprint2_output["research_chunks"][:5]

pathlib.Path("sprint2_output.json").write_text(
    json.dumps(sprint2_output, indent=2, default=str))
print("✅ Sprint 2 output saved to sprint2_output.json")


Pinecone namespace 'anthropic': 153 vectors
Vectors this run   : 7

⚠️  Only 7 vectors — re-chunking with smaller size...
   Re-chunked into 35 pieces
   ✅ Total vectors now: 42

Pinecone namespace 'anthropic': 188 vectors total

✅ Status   : stored (need 'stored')
✅ Vectors  : 42 (need ≥10)

✅ Sprint 2 DoD: MET — ≥10 vectors in Pinecone
🎉 Ready for Sprint 3
✅ Sprint 2 output saved to sprint2_output.json
